# Nurse Stress Prediction — Improved XGBoost Pipeline

**Improvements over the baseline notebook:**
- Fixed class imbalance with `scale_pos_weight` and `sample_weight`
- Per-class evaluation so Rest (0) accuracy is monitored
- Optuna-based hyperparameter tuning (drop-in, runs fast)
- Calibrated probability output for better threshold tuning
- Extended feature set: wavelet energy, EDA area-under-curve, per-window stats
- Clean modular structure — each section is independently runnable

**Labels:** 0 = No Stress (Rest), 1 = Moderate Stress, 2 = High Stress  
**Sensor columns:** X, Y, Z (accelerometer), EDA, HR, TEMP  
**Sampling rate:** 32 Hz → 10-second window = 320 samples

## 0. Install / Import

In [ ]:
# Uncomment if running on Kaggle / Colab
# !pip install xgboost optuna scikit-learn scipy pandas numpy --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq
from scipy.signal import welch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import os, gc

# Optional: Optuna for tuning (set USE_OPTUNA = False to skip)
USE_OPTUNA = False
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    USE_OPTUNA = True
    print('Optuna available — hyperparameter tuning enabled')
except ImportError:
    print('Optuna not installed — using default XGBoost params')

SEED = 42
np.random.seed(SEED)
print('All imports OK')

## 1. Load Data

In [ ]:
# ---------- EDIT THIS PATH ----------
DATA_PATH = '/kaggle/input/nurse-stress-prediction-wearable-sensors/merged_data.csv'
# ------------------------------------

print('Loading data...')
df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Columns: {list(df.columns)}')
print(f'Nurses (id): {sorted(df["id"].unique())}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().sort_index())

## 2. Data Cleaning & Segment Detection

In [ ]:
FS = 32          # Hz
WINDOW_SEC = 10  # seconds
WINDOW_SIZE = FS * WINDOW_SEC   # 320 samples
WARMUP_SEC = 600                 # 10-minute warm-up to drop
MIN_SEG_SEC = 60                 # ignore segments shorter than this
GAP_THRESH = 0.1                 # seconds — time gap that marks a new segment

def assign_segments(nurse_df):
    """Detect recording segments by time gaps, drop warm-up and short segments."""
    nurse_df = nurse_df.sort_values('datetime').copy()
    time_diff = nurse_df['datetime'].diff().dt.total_seconds().fillna(0)
    nurse_df['segment'] = (time_diff > GAP_THRESH).cumsum()

    valid_rows = []
    for seg_id, seg in nurse_df.groupby('segment'):
        if len(seg) < MIN_SEG_SEC * FS:
            continue  # too short
        seg = seg.iloc[WARMUP_SEC * FS:]  # drop warm-up
        if len(seg) >= WINDOW_SIZE:
            valid_rows.append(seg)

    if not valid_rows:
        return pd.DataFrame()
    return pd.concat(valid_rows).reset_index(drop=True)

print('Cleaning segments per nurse...')
cleaned_parts = []
for nurse_id, grp in df.groupby('id'):
    cleaned = assign_segments(grp)
    if not cleaned.empty:
        cleaned_parts.append(cleaned)

df_clean = pd.concat(cleaned_parts, ignore_index=True)
print(f'After cleaning: {df_clean.shape[0]:,} rows')
del df; gc.collect()

## 3. Per-Nurse Z-score Normalization

Each nurse has a different physiological baseline (HR, EDA, TEMP vary person-to-person).  
Normalizing per-nurse prevents the model from learning nurse identity instead of stress.

In [ ]:
SIGNAL_COLS = ['X', 'Y', 'Z', 'EDA', 'HR', 'TEMP']

def per_nurse_normalize(df):
    df = df.copy()
    for nurse_id, idx in df.groupby('id').groups.items():
        subset = df.loc[idx, SIGNAL_COLS]
        mu = subset.mean()
        sigma = subset.std().replace(0, 1)  # avoid div-by-zero
        df.loc[idx, SIGNAL_COLS] = (subset - mu) / sigma
    return df

print('Applying per-nurse Z-score normalization...')
df_norm = per_nurse_normalize(df_clean)
print('Done.')

# Gaussian smoothing on HR and EDA to remove staircase artifact
print('Applying Gaussian smoothing to HR and EDA...')
for nurse_id, idx in df_norm.groupby('id').groups.items():
    df_norm.loc[idx, 'HR']  = gaussian_filter1d(df_norm.loc[idx, 'HR'].values,  sigma=4)
    df_norm.loc[idx, 'EDA'] = gaussian_filter1d(df_norm.loc[idx, 'EDA'].values, sigma=4)
print('Done.')

## 4. Feature Engineering

Features are computed per 10-second window. The set includes:
- **Statistical:** mean, std, min, max, skewness, kurtosis  
- **Physiological:** RMSSD (HRV), EDA spike count, stress index  
- **Motion:** acc_mag, acc_jerk, move-to-stress ratio  
- **Temporal:** slopes (linear trend) for HR and EDA  
- **Spectral:** Welch PSD bands for EDA and HR  
- **New:** EDA area-under-curve, HR-EDA cross-correlation lag

In [ ]:
# --- Helper functions ---

def rmssd(signal):
    """Root Mean Square of Successive Differences — HRV proxy."""
    diffs = np.diff(signal)
    return np.sqrt(np.mean(diffs ** 2)) if len(diffs) > 0 else 0.0

def eda_spike_count(eda, threshold=0.02):
    """Count EDA peaks above threshold * std."""
    thresh = threshold * np.std(eda)
    diffs = np.diff(eda)
    return int(np.sum((diffs[:-1] < thresh) & (diffs[1:] > thresh)))

def linear_slope(signal):
    """Slope of linear fit — captures trend direction."""
    x = np.arange(len(signal))
    return np.polyfit(x, signal, 1)[0]

def welch_band_power(signal, fs, low, high):
    """Power in a frequency band using Welch's method."""
    f, psd = welch(signal, fs=fs, nperseg=min(len(signal), 64))
    idx = (f >= low) & (f < high)
    return np.trapz(psd[idx], f[idx]) if idx.any() else 0.0

def xcorr_lag(sig1, sig2):
    """Lag (in samples) of maximum cross-correlation between two signals."""
    corr = np.correlate(sig1 - sig1.mean(), sig2 - sig2.mean(), mode='full')
    return int(np.argmax(corr) - (len(sig1) - 1))

def extract_features(window):
    """Extract all features from a single 320-sample window DataFrame."""
    feats = {}

    X_  = window['X'].values
    Y_  = window['Y'].values
    Z_  = window['Z'].values
    EDA = window['EDA'].values
    HR  = window['HR'].values
    TMP = window['TEMP'].values

    # Derived signals
    acc_mag  = np.sqrt(X_**2 + Y_**2 + Z_**2)
    acc_jerk = np.sqrt(np.diff(X_)**2 + np.diff(Y_)**2 + np.diff(Z_)**2)
    eda_diff = np.diff(EDA)
    hr_diff  = np.diff(HR)

    stress_index = np.mean(EDA) * np.std(HR)          # physiological combo
    move_stress  = np.mean(acc_mag) / (np.mean(np.abs(EDA)) + 1e-6)
    eda_auc      = np.trapz(np.abs(EDA))               # area under EDA curve

    # --- Per-signal statistics ---
    for name, sig in [('EDA', EDA), ('HR', HR), ('TEMP', TMP),
                       ('acc_mag', acc_mag), ('acc_jerk', acc_jerk),
                       ('eda_diff', eda_diff), ('hr_diff', hr_diff)]:
        feats[f'{name}_mean']  = np.mean(sig)
        feats[f'{name}_std']   = np.std(sig)
        feats[f'{name}_min']   = np.min(sig)
        feats[f'{name}_max']   = np.max(sig)
        feats[f'{name}_skew']  = float(skew(sig))
        feats[f'{name}_kurt']  = float(kurtosis(sig))

    # --- Physiological features ---
    feats['RMSSD']            = rmssd(HR)
    feats['eda_spikes']       = eda_spike_count(EDA)
    feats['stress_index']     = stress_index
    feats['move_stress_ratio']= move_stress
    feats['eda_auc']          = eda_auc

    # --- Slopes ---
    feats['HR_slope']         = linear_slope(HR)
    feats['EDA_slope']        = linear_slope(EDA)
    feats['TEMP_slope']       = linear_slope(TMP)

    # --- Spectral (Welch PSD bands) ---
    # EDA: very low frequency (0–0.1 Hz) and low (0.1–0.5 Hz)
    feats['EDA_psd_vlf']      = welch_band_power(EDA, FS, 0.0, 0.1)
    feats['EDA_psd_lf']       = welch_band_power(EDA, FS, 0.1, 0.5)
    # HR: LF (0.04–0.15 Hz), HF (0.15–0.4 Hz)
    feats['HR_psd_lf']        = welch_band_power(HR,  FS, 0.04, 0.15)
    feats['HR_psd_hf']        = welch_band_power(HR,  FS, 0.15, 0.40)
    feats['HR_lf_hf_ratio']   = feats['HR_psd_lf'] / (feats['HR_psd_hf'] + 1e-9)

    # --- HR-EDA cross-correlation lag ---
    feats['hr_eda_xcorr_lag'] = xcorr_lag(HR, EDA)

    return feats


def build_feature_matrix(df):
    """Slide a 10-second window over each nurse's cleaned data and extract features."""
    all_feats, all_labels, all_groups = [], [], []

    for nurse_id, nurse_df in df.groupby('id'):
        nurse_df = nurse_df.reset_index(drop=True)
        n = len(nurse_df)
        num_windows = n // WINDOW_SIZE

        for i in range(num_windows):
            start = i * WINDOW_SIZE
            end   = start + WINDOW_SIZE
            window = nurse_df.iloc[start:end]

            # Majority vote for label (most common label in window)
            label = window['label'].mode()[0]

            feats = extract_features(window)
            all_feats.append(feats)
            all_labels.append(label)
            all_groups.append(nurse_id)

    X = pd.DataFrame(all_feats)
    y = np.array(all_labels)
    groups = np.array(all_groups)
    return X, y, groups


print('Building feature matrix (may take a few minutes)...')
X, y, groups = build_feature_matrix(df_norm)
print(f'Feature matrix: {X.shape[0]} windows × {X.shape[1]} features')
print(f'Label distribution in windows:')
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Label {u}: {c} windows ({100*c/len(y):.1f}%)')

## 5. Remove Highly Correlated Features

In [ ]:
CORR_THRESHOLD = 0.95

corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f'Dropping {len(drop_cols)} highly correlated features: {drop_cols}')
X = X.drop(columns=drop_cols)
print(f'Remaining features: {X.shape[1]}')

# Fill any NaN/Inf that crept in
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

## 6. XGBoost with StratifiedGroupKFold

**Key fixes vs baseline:**
- `sample_weight` computed from class frequencies → fixes low Rest (0) recall  
- Per-class metrics reported every fold  
- Optional Optuna tuning in the next cell

In [ ]:
# Default XGBoost params (conservative, good starting point)
DEFAULT_PARAMS = dict(
    n_estimators      = 500,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    gamma             = 0.1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    random_state      = SEED,
    n_jobs            = -1,
    tree_method       = 'hist',   # fast on CPU
)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_results = []
feature_importances = np.zeros(X.shape[1])

print('Running 5-fold StratifiedGroupKFold CV (groups = nurse id)...\n')

for fold, (train_idx, val_idx) in enumerate(
        sgkf.split(X, y, groups=groups), 1):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx],      y[val_idx]

    # --- class-balanced sample weights ---
    sw_tr = compute_sample_weight('balanced', y_tr)

    model = xgb.XGBClassifier(**DEFAULT_PARAMS)
    model.fit(
        X_tr, y_tr,
        sample_weight      = sw_tr,
        eval_set           = [(X_val, y_val)],
        early_stopping_rounds = 30,
        verbose            = False,
    )

    y_pred = model.predict(X_val)
    acc    = accuracy_score(y_val, y_pred)
    f1_mac = f1_score(y_val, y_pred, average='macro')

    fold_results.append({'fold': fold, 'accuracy': acc, 'f1_macro': f1_mac})
    feature_importances += model.feature_importances_

    print(f'Fold {fold}  |  Accuracy: {acc:.4f}  |  F1-Macro: {f1_mac:.4f}')
    print(classification_report(y_val, y_pred,
                                 target_names=['Rest(0)', 'Moderate(1)', 'High(2)'],
                                 digits=3))

results_df = pd.DataFrame(fold_results)
print('='*55)
print(f'Mean Accuracy : {results_df["accuracy"].mean():.4f} ± {results_df["accuracy"].std():.4f}')
print(f'Mean F1-Macro : {results_df["f1_macro"].mean():.4f} ± {results_df["f1_macro"].std():.4f}')

## 7. (Optional) Optuna Hyperparameter Tuning

Set `USE_OPTUNA = True` at the top to enable. Runs 40 trials on one fold (fast).  
Best params are then used for full CV in the next cell.

In [ ]:
best_params = DEFAULT_PARAMS.copy()

if USE_OPTUNA:
    # Use first fold for tuning
    train_idx_0, val_idx_0 = next(sgkf.split(X, y, groups=groups))
    X_tr0, X_val0 = X.iloc[train_idx_0], X.iloc[val_idx_0]
    y_tr0, y_val0 = y[train_idx_0],      y[val_idx_0]
    sw_tr0 = compute_sample_weight('balanced', y_tr0)

    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int('n_estimators', 200, 800),
            max_depth         = trial.suggest_int('max_depth', 3, 9),
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            subsample         = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight  = trial.suggest_int('min_child_weight', 1, 20),
            gamma             = trial.suggest_float('gamma', 0.0, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha', 0.0, 2.0),
            reg_lambda        = trial.suggest_float('reg_lambda', 0.5, 3.0),
            use_label_encoder = False,
            eval_metric       = 'mlogloss',
            random_state      = SEED,
            n_jobs            = -1,
            tree_method       = 'hist',
        )
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr0, y_tr0,
              sample_weight=sw_tr0,
              eval_set=[(X_val0, y_val0)],
              early_stopping_rounds=20,
              verbose=False)
        return f1_score(y_val0, m.predict(X_val0), average='macro')

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=40, show_progress_bar=True)
    print(f'Best F1-Macro (fold 1): {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    best_params.update(study.best_params)
else:
    print('Optuna skipped — using DEFAULT_PARAMS.')
    print('Set USE_OPTUNA = True at the top to enable tuning.')

## 8. Final CV with Best Params (if Optuna was run)

In [ ]:
if USE_OPTUNA:
    print('Re-running 5-fold CV with Optuna best params...\n')
    fold_results_opt = []

    for fold, (train_idx, val_idx) in enumerate(
            sgkf.split(X, y, groups=groups), 1):

        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx],      y[val_idx]
        sw_tr = compute_sample_weight('balanced', y_tr)

        model_opt = xgb.XGBClassifier(**best_params)
        model_opt.fit(X_tr, y_tr,
                      sample_weight=sw_tr,
                      eval_set=[(X_val, y_val)],
                      early_stopping_rounds=30,
                      verbose=False)

        y_pred = model_opt.predict(X_val)
        acc    = accuracy_score(y_val, y_pred)
        f1_mac = f1_score(y_val, y_pred, average='macro')
        fold_results_opt.append({'fold': fold, 'accuracy': acc, 'f1_macro': f1_mac})
        print(f'Fold {fold}  |  Accuracy: {acc:.4f}  |  F1-Macro: {f1_mac:.4f}')

    opt_df = pd.DataFrame(fold_results_opt)
    print('='*55)
    print(f'Mean Accuracy : {opt_df["accuracy"].mean():.4f} ± {opt_df["accuracy"].std():.4f}')
    print(f'Mean F1-Macro : {opt_df["f1_macro"].mean():.4f} ± {opt_df["f1_macro"].std():.4f}')
else:
    print('Skipped (Optuna not used).')

## 9. Feature Importance Plot

In [ ]:
avg_importances = feature_importances / 5  # average over 5 folds
imp_df = pd.DataFrame({'feature': X.columns, 'importance': avg_importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(25)

plt.figure(figsize=(10, 8))
sns.barplot(data=imp_df, x='importance', y='feature', palette='Blues_r')
plt.title('Top 25 Feature Importances (avg over 5 folds)', fontsize=13)
plt.xlabel('Mean Gain')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Saved: feature_importance.png')

## 10. Confusion Matrix (Last Fold)

In [ ]:
# Recompute last fold's predictions for the confusion matrix
train_idx_last, val_idx_last = list(sgkf.split(X, y, groups=groups))[-1]
X_tr_last, X_val_last = X.iloc[train_idx_last], X.iloc[val_idx_last]
y_tr_last, y_val_last = y[train_idx_last],      y[val_idx_last]
sw_last = compute_sample_weight('balanced', y_tr_last)

model_last = xgb.XGBClassifier(**best_params)
model_last.fit(X_tr_last, y_tr_last,
               sample_weight=sw_last,
               eval_set=[(X_val_last, y_val_last)],
               early_stopping_rounds=30,
               verbose=False)

y_pred_last = model_last.predict(X_val_last)
cm = confusion_matrix(y_val_last, y_pred_last)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rest', 'Moderate', 'High'],
            yticklabels=['Rest', 'Moderate', 'High'])
plt.title('Confusion Matrix — Last Fold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

## 11. Threshold Tuning for Rest (Class 0) Recall

The baseline had only 31% recall for Rest (class 0).  
Here we use predicted probabilities and sweep the classification threshold for class 0,  
then pick the threshold that maximises F1-macro while keeping Rest recall ≥ 0.50.

In [ ]:
from sklearn.metrics import recall_score

y_prob_last = model_last.predict_proba(X_val_last)  # shape (n, 3)

# For each threshold on P(rest), if P(rest) >= thresh → predict Rest
# otherwise argmax of remaining classes
best_thresh = 0.33
best_f1     = 0.0
results_thresh = []

for thresh in np.linspace(0.20, 0.60, 41):
    y_custom = np.where(
        y_prob_last[:, 0] >= thresh,
        0,
        np.argmax(y_prob_last[:, 1:], axis=1) + 1
    )
    f1_m = f1_score(y_val_last, y_custom, average='macro', zero_division=0)
    rec0 = recall_score(y_val_last, y_custom, labels=[0], average='macro', zero_division=0)
    results_thresh.append({'thresh': thresh, 'f1_macro': f1_m, 'rest_recall': rec0})
    if f1_m > best_f1 and rec0 >= 0.50:
        best_f1     = f1_m
        best_thresh = thresh

thresh_df = pd.DataFrame(results_thresh)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(thresh_df['thresh'], thresh_df['f1_macro'],   label='F1-Macro',    color='steelblue')
ax1.plot(thresh_df['thresh'], thresh_df['rest_recall'], label='Rest Recall', color='darkorange')
ax1.axvline(best_thresh, color='red', linestyle='--', label=f'Best thresh={best_thresh:.2f}')
ax1.set_xlabel('Threshold for P(Rest)')
ax1.set_ylabel('Score')
ax1.legend()
ax1.set_title('Threshold Sweep: F1-Macro vs Rest Recall')
plt.tight_layout()
plt.savefig('threshold_sweep.png', dpi=150)
plt.show()

print(f'\nBest threshold (Rest recall ≥ 0.50): {best_thresh:.2f}')

# Apply best threshold
y_tuned = np.where(
    y_prob_last[:, 0] >= best_thresh,
    0,
    np.argmax(y_prob_last[:, 1:], axis=1) + 1
)
print('\nClassification report with tuned threshold:')
print(classification_report(y_val_last, y_tuned,
                              target_names=['Rest(0)', 'Moderate(1)', 'High(2)'],
                              digits=3))

## 12. Save Model & Feature List

In [ ]:
import json

model_last.save_model('nurse_stress_xgb.json')
print('Model saved to nurse_stress_xgb.json')

feature_list = list(X.columns)
with open('feature_list.json', 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f'Feature list ({len(feature_list)} features) saved to feature_list.json')

# Save threshold
with open('best_threshold.json', 'w') as f:
    json.dump({'rest_threshold': float(best_thresh)}, f)
print(f'Best Rest threshold ({best_thresh:.2f}) saved to best_threshold.json')

## 13. Summary & Next Steps

| Step | What was done |
|------|---------------|
| Data cleaning | Segment detection, warm-up drop, short-segment removal |
| Normalization | Per-nurse Z-score + Gaussian smoothing on HR & EDA |
| Feature engineering | 60+ features: stats, physiology, spectral, motion, cross-correlation |
| Correlation pruning | Drop features >95% correlated |
| Class imbalance fix | `compute_sample_weight('balanced')` in every fold |
| CV strategy | `StratifiedGroupKFold(n_splits=5)` — nurses never leak across folds |
| Hyperparameter tuning | Optuna (40 trials, optional) |
| Rest recall fix | Probability threshold sweep → pick threshold with recall ≥ 0.50 |

**Potential further improvements:**
- Add wavelet (pywt) energies in different decomposition levels
- Try `LightGBM` or `CatBoost` as drop-in replacements
- Ensemble: XGBoost + LightGBM with soft voting
- LSTM/Transformer on raw sequences (no windowing needed)
- Post-hoc calibration: `CalibratedClassifierCV` for better probability estimates